In [ ]:
import re
import pandas as pd

In [ ]:
df = pd.read_csv("../data/raw/batdongsan_com_vn.csv")

In [ ]:
df.info()

# Extract numerics

In [ ]:
def extract_numeric(s:str|None) -> float:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^(\d+(?:.\d+)*,?\d*)\D?", s)
    if not results:
        return None
    else:
        num_str = results.group(1)
        num_str = num_str.replace(".","")
        num_str = num_str.replace(",",".")
        return float(num_str)
    
def extract_measuring_unit(s:str|None) -> str:
    if not isinstance(s,str) or not s:
        return None

    results = re.search(r"^\d+(?:.\d+)*,?\d*\s*(\D*)", s)
    if not results:
        return None
    else:
        return results.group(1)

In [ ]:
df["price_val"] = df.price.apply(extract_numeric)
df["price_unit"] = df.price.apply(extract_measuring_unit)
df["area_val"] = df.area.apply(extract_numeric)
df["area_unit"] = df.area.apply(extract_measuring_unit)
df["n_bedrooms_val"] = df.n_bedrooms.apply(extract_numeric) 
df["n_bathrooms_val"] = df.n_bathrooms.apply(extract_numeric)
df["n_floors_val"] = df.n_floors.apply(extract_numeric)
df["n_floors_val"] = df.n_floors.apply(extract_numeric)
df["front_width_val"] = df.front_width.apply(extract_numeric)
df["front_width_unit"] = df.front_width.apply(extract_measuring_unit)
df["front_road_width_val"] = df.front_road_width.apply(extract_numeric)
df["front_road_width_unit"] = df.front_road_width.apply(extract_measuring_unit)

df[[
    "price_val", "price_unit", "area_val", "area_unit", "n_bedrooms_val", "n_bathrooms_val", "n_floors_val", 
    "front_width_val", "front_width_unit", "front_road_width_val", "front_road_width_unit"
]].describe(include="all")

From this table, there"s 2 noticeable problems:
- The non-null count drops <- "Thỏa thuận" prices got converted None
- 6 unique, non-null `price_unit` -> Price unit inconsistency  

For the first problem, we'll analyze the observations with "Thỏa thuận" prices 
separately and use the rest for the regression model.  
For the second problem, we just need to standardize the units 

# Extract city, district

In [ ]:
def extract_location_detail(addr: str, level:int) -> str|None:
    '''
    level:int - the level of details to extract from the address. 1 -> Province; 2 -> District
    '''
    if not isinstance(addr, str) or not addr.strip():
        return None
    parts = [p.strip() for p in addr.split(",") if p.strip() != ""]
    if len(parts) >= level:
        result = parts[-level].replace('.','')
        return result
    return None

In [ ]:
df['city_province'] = df.location.apply(extract_location_detail, level = 1)
df['district'] = df.location.apply(extract_location_detail, level = 2)
df[['city_province', 'district', 'location']].sample(5)

# Select columns for processing

In [ ]:
df.info()

In [ ]:
cols_for_analysis = [
    # highly relevant for analysis
    'price_val', 'price_unit', 'area_val', 'area_unit', 'n_bedrooms_val', 'n_bathrooms_val', 'n_floors_val','front_width_val', 'front_width_unit', 'front_road_width_val', 'front_road_width_unit',
    'legal', 'facing_direction', 'balcony_direction', 'city_province', 'district', 'property_type',

    # might be useful later extractions
    'latitude', 'longitude', 'title', 'description', 'location_details', 'date_of_posting', 'interior', 'verified',

    # raw for later checks
    'price', 'area', 'n_bedrooms', 'n_bathrooms', 'front_width', 'front_road_width', 'n_floors'
]
df_pruned = df[cols_for_analysis]
df_pruned = df_pruned.rename({
    'price_val': 'price',
    'area_val':'area',
    'front_width_val':'front_width',
    'front_road_width_val':'front_road_width',
    'legal':'legal_docs',
    'location_details':'address',
    'price':'raw_price',
    'area':'raw_area',
    'front_width': 'raw_front_width',
    'front_road_width': 'raw_front_road_width',
    'n_bedrooms_val':'n_bedrooms',
    'n_bathrooms_val':'n_bathrooms',
    'n_bedrooms':'raw_n_bedrooms',
    'n_bathrooms':'raw_n_bathrooms',
    'n_floors_val':'n_floors',
    'n_floors':'raw_n_floors'
}, axis=1)
df_pruned['scraper'] = 'Quang'
df_pruned.sample(3)

In [ ]:
df_pruned.to_csv('../data/interim/batdongsan_com_vn(1).csv')